# Learning recursions

This first exercise is a quick introduction to RNNs as models able to learn recursion patterns in sequential data.

In this exercise, we will use the same `torch` functions and framework as before

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt

We would like to see if a simple RNN can learn the recursion pattern of the famous Fibonacci sequence. 
We thus create below a dataset where inputs `X` are sub-sequences of the Fibonacci sequence while targets `y` are the next value of the sequence.

In [2]:
fibonacci = np.array([0,1,1,2,3,5,8,13,21,34,55,89,144,233,377,610,987])
d_inputs = 1 #input sesquence element dimensionality
n_steps = 2 #sequence length
n = len(fibonacci) - n_steps #number of sequences
X = np.zeros((n,n_steps,d_inputs),dtype=np.float32)
y = np.zeros((n,d_inputs),dtype=np.float32)
for i in range(n):
    X[i,:,0] = fibonacci[i:i+n_steps]
    y[i] = fibonacci[i+n_steps]

print(X.reshape((n,n_steps)))
print(y)

[[  0.   1.]
 [  1.   1.]
 [  1.   2.]
 [  2.   3.]
 [  3.   5.]
 [  5.   8.]
 [  8.  13.]
 [ 13.  21.]
 [ 21.  34.]
 [ 34.  55.]
 [ 55.  89.]
 [ 89. 144.]
 [144. 233.]
 [233. 377.]
 [377. 610.]]
[[  1.]
 [  2.]
 [  3.]
 [  5.]
 [  8.]
 [ 13.]
 [ 21.]
 [ 34.]
 [ 55.]
 [ 89.]
 [144.]
 [233.]
 [377.]
 [610.]
 [987.]]


As usual, we shall begin our computational graph by creating appropriate `tf.placeholder` instances.

**Q1** Create two `Tensor` instances with same shapes as `X` and `y`.

In [3]:
batch_size=n
Xt = torch.from_numpy(X).reshape(batch_size, n_steps, d_inputs)  
yt = torch.from_numpy(y).reshape(batch_size, d_inputs)   
print(Xt.shape,yt.shape)

torch.Size([15, 2, 1]) torch.Size([15, 1])


A sequence length of 4 has be chosen. We need to design a simple RNN architecture that has the capacity to learn that the target is equal to the sum of the last and penultimate elements of the input sequence.

We thus investigate a two-layer architecture:
* Layer 1 is a simple RNN layer with 2 neural units and thus will memorize a hidden state of length 2,
* Layer 2 is a fully connected layer that should learn to add.

Both layers will use linear activiations.

The code below creates `torch.Tensor` instances for the first layer. Remember that at $t=0$, Layer 1 must compute

$$ \mathbf{h}_0 = \mathbf{W}_x \cdot \mathbf{x}_0 + b. \text{ (1)}$$

Then, at every other time step, Layer 1 must compute

$$ \mathbf{h}_t = \mathbf{W}_x \cdot \mathbf{x}_t + \mathbf{W}_h \cdot \mathbf{h}_{t-1} + b. \text{ (2)}$$

The code below creates `torch.Tensor` instances for the trainable parameters of Layer 1

In [4]:
n_hs_neurons = 2 #number of neural units in the simple RNN cell

# Wx maps an input x_t of shape (batch_size, d_inputs) to the hidden space (batch_size, n_hs_neurons)
Wx = nn.Parameter(torch.randn(d_inputs, n_hs_neurons, dtype=torch.float32))

# Wh maps the previous hidden state h_{t-1} of shape (batch_size, n_hs_neurons) to the next hidden state
Wh = nn.Parameter(torch.randn(n_hs_neurons, n_hs_neurons, dtype=torch.float32))

# Bias is broadcasted over the batch dimension
b = nn.Parameter(torch.zeros(1, n_hs_neurons, dtype=torch.float32))


Now, to create the RNN layer, torch actully creates a computational graph corresponding to the unrolled version of the RNN layer. So, we need to separate our dataset w.r.t. time steps. This consists in slicing the input tensor w.r.t. the time dimension (axis 1 in our case). Then each portion of the input tensor is used separately in the graph !

The `torch.unbind` does this job:

In [5]:
X_seqs = torch.unbind(Xt,dim=1) 
X_seqs

(tensor([[  0.],
         [  1.],
         [  1.],
         [  2.],
         [  3.],
         [  5.],
         [  8.],
         [ 13.],
         [ 21.],
         [ 34.],
         [ 55.],
         [ 89.],
         [144.],
         [233.],
         [377.]]),
 tensor([[  1.],
         [  1.],
         [  2.],
         [  3.],
         [  5.],
         [  8.],
         [ 13.],
         [ 21.],
         [ 34.],
         [ 55.],
         [ 89.],
         [144.],
         [233.],
         [377.],
         [610.]]))

`X_seqs` is a list of tf tensors of shapes `(n,d_inputs)`, i.e. (13,1) in our case. For instance `X_seqs[0]` is a tensor that will contain each first sequence element of the training sequences, or in other words, the first column of the above printed `X`.

Now comes the computation part. We need to create the graph by first computing $\mathbf{h}_0$. Using `torch.matmul` this is quite simple. Then we need to loop on the remaining time steps, compute the current hidden state $\mathbf{h}_t$ using equation (2) and append this graph node to the list `hidden_states` (because the newly created node will be used in the next iteration).

**Q2** Fill the gaps in the code below to create the unrolled simple RNN layer.

In [6]:
h0 = torch.matmul(X_seqs[0], Wx) + b
hidden_states = [h0]

print(hidden_states[-1].shape)

for i in range(1, n_steps):
    XWx = torch.matmul(X_seqs[i], Wx)
    HWh = torch.matmul(hidden_states[-1], Wh)
    hidden_states.append(XWx + HWh + b)

# last hidden state is the representation of the full input sequence
hidden_states[-1]


torch.Size([15, 2])


tensor([[3.7316e-01, 1.7265e-01],
        [3.5755e-01, 1.6437e-01],
        [7.3070e-01, 3.3701e-01],
        [1.0882e+00, 5.0138e-01],
        [1.8190e+00, 8.3840e-01],
        [2.9072e+00, 1.3398e+00],
        [4.7262e+00, 2.1782e+00],
        [7.6334e+00, 3.5180e+00],
        [1.2360e+01, 5.6961e+00],
        [1.9993e+01, 9.2141e+00],
        [3.2352e+01, 1.4910e+01],
        [5.2345e+01, 2.4124e+01],
        [8.4698e+01, 3.9035e+01],
        [1.3704e+02, 6.3159e+01],
        [2.2174e+02, 1.0219e+02]], grad_fn=<AddBackward0>)

The last hidden state cannot be used as output $\hat{y}$ because it has size 2. So we need a second (fully connected layer) to compute a one-dimensional output from the last hidden state.

**Q3** Fill the gaps in the code below to create the fully connected layer that complete the network.

In [7]:
mlp = torch.nn.Linear(in_features=n_hs_neurons, out_features=1)
output = mlp(hidden_states[-1])

mseloss = torch.nn.MSELoss()
print("yt:", yt.shape, "| output:", output.shape)
loss = mseloss(output, yt)
print("Initial loss:", loss.item())


yt: torch.Size([15, 1]) | output: torch.Size([15, 1])
Initial loss: 119736.640625


Let's create a module regrouping all the components !

In [8]:
class SimpleRNN(torch.nn.Module):
    def __init__(self, n_hs_neurons, n_steps):
        super().__init__()
        self.n_hs_neurons = n_hs_neurons
        self.n_steps = n_steps

        self.Wx = nn.Parameter(torch.randn(d_inputs, n_hs_neurons, dtype=torch.float32))
        self.Wh = nn.Parameter(torch.randn(n_hs_neurons, n_hs_neurons, dtype=torch.float32))
        self.b = nn.Parameter(torch.zeros(1, n_hs_neurons, dtype=torch.float32))

        # Final linear layer that maps the last hidden state to the 1D output
        self.mlp = torch.nn.Linear(in_features=n_hs_neurons, out_features=1)

    def forward(self, x):
        # x: (batch_size, n_steps, d_inputs)
        X_seqs = torch.unbind(x, dim=1)

        h = torch.matmul(X_seqs[0], self.Wx) + self.b
        for i in range(1, self.n_steps):
            h = torch.matmul(X_seqs[i], self.Wx) + torch.matmul(h, self.Wh) + self.b

        return self.mlp(h)


We have also added a loss to the graph. We can now train the model and check if it works.

In [9]:
learning_rate = 0.001
epochs = 800

mseloss = torch.nn.MSELoss()
mlp = torch.nn.Linear(in_features=n_hs_neurons, out_features=1)
optimizer = torch.optim.Adam([Wx, Wh, b, mlp.weight, mlp.bias], lr=learning_rate)

for epoch in range(epochs):
    optimizer.zero_grad()

    h0 = torch.matmul(X_seqs[0], Wx) + b
    hidden_states = [h0]

    for i in range(1, n_steps):
        XWx = torch.matmul(X_seqs[i], Wx)
        HWh = torch.matmul(hidden_states[-1], Wh)
        hidden_states.append(XWx + HWh + b)

    output = mlp(hidden_states[-1])
    loss = mseloss(output, yt)

    loss.backward()
    optimizer.step()

    if epoch % 200 == 199:
        print(f"Epoch {epoch:4d} | Loss {loss.item():.6f}")


Epoch  199 | Loss 78661.445312
Epoch  399 | Loss 12992.890625
Epoch  599 | Loss 31.456200


Epoch  799 | Loss 0.945629


In [10]:
learning_rate = 0.001
epochs = 2000

mseloss = torch.nn.MSELoss()
model = SimpleRNN(n_hs_neurons, n_steps)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
model.train()

for epoch in range(epochs):
    optimizer.zero_grad()
    output = model(Xt)
    loss = mseloss(output, yt)
    loss.backward()
    optimizer.step()

    if epoch % 400 == 399:
        model.eval()
        abs_err = (yt.flatten().detach() - output.flatten().detach())
        rel_err = abs_err / yt.flatten().detach()
        print("abs error:", abs_err)
        print("rel error:", rel_err)
        print(f"Epoch {epoch} | Loss {loss.item():.8f}\n")
        model.train()


abs error: tensor([-0.5975,  0.8120, -0.0598,  0.4778,  0.1436,  0.3470,  0.2163,  0.2890,
         0.2310,  0.2457,  0.2023,  0.1737,  0.1017,  0.0010, -0.1716])
rel error: tensor([-5.9748e-01,  4.0598e-01, -1.9949e-02,  9.5556e-02,  1.7950e-02,
         2.6696e-02,  1.0300e-02,  8.5006e-03,  4.2000e-03,  2.7605e-03,
         1.4052e-03,  7.4545e-04,  2.6980e-04,  1.6009e-06, -1.7389e-04])
Epoch 399 | Loss 0.11627587

abs error: tensor([-0.6113,  0.7982, -0.0736,  0.4640,  0.1299,  0.3335,  0.2030,  0.2760,
         0.2184,  0.2339,  0.1919,  0.1653,  0.0967,  0.0015, -0.1624])
rel error: tensor([-6.1128e-01,  3.9908e-01, -2.4539e-02,  9.2809e-02,  1.6243e-02,
         2.5654e-02,  9.6646e-03,  8.1168e-03,  3.9717e-03,  2.6283e-03,
         1.3324e-03,  7.0937e-04,  2.5653e-04,  2.4014e-06, -1.6455e-04])
Epoch 799 | Loss 0.11194837



abs error: tensor([-0.6298,  0.7796, -0.0921,  0.4456,  0.1116,  0.3153,  0.1850,  0.2584,
         0.2016,  0.2181,  0.1777,  0.1539,  0.0897,  0.0017, -0.1506])
rel error: tensor([-6.2980e-01,  3.8981e-01, -3.0698e-02,  8.9123e-02,  1.3952e-02,
         2.4255e-02,  8.8107e-03,  7.6011e-03,  3.6647e-03,  2.4503e-03,
         1.2342e-03,  6.6052e-04,  2.3791e-04,  2.8016e-06, -1.5256e-04])
Epoch 1199 | Loss 0.10657232

abs error: tensor([-0.6520,  0.7574, -0.1142,  0.4235,  0.0896,  0.2935,  0.1635,  0.2374,
         0.1813,  0.1991,  0.1607,  0.1402,  0.0813,  0.0019, -0.1365])
rel error: tensor([-6.5200e-01,  3.7869e-01, -3.8081e-02,  8.4702e-02,  1.1205e-02,
         2.2578e-02,  7.7867e-03,  6.9826e-03,  3.2963e-03,  2.2368e-03,
         1.1161e-03,  6.0158e-04,  2.1557e-04,  3.1018e-06, -1.3833e-04])
Epoch 1599 | Loss 0.10077728



abs error: tensor([-0.6768,  0.7325, -0.1390,  0.3988,  0.0651,  0.2691,  0.1395,  0.2139,
         0.1586,  0.1778,  0.1417,  0.1248,  0.0718,  0.0019, -0.1210])
rel error: tensor([-6.7680e-01,  3.6626e-01, -4.6335e-02,  7.9757e-02,  8.1328e-03,
         2.0702e-02,  6.6416e-03,  6.2907e-03,  2.8844e-03,  1.9979e-03,
         9.8409e-04,  5.3576e-04,  1.9055e-04,  3.1018e-06, -1.2263e-04])
Epoch 1999 | Loss 0.09514032



In [11]:
print(model.Wh, model.Wx, model.b)

Parameter containing:
tensor([[-0.1232,  0.2898],
        [-0.2724, -0.0925]], requires_grad=True) Parameter containing:
tensor([[ 1.8425, -0.8870]], requires_grad=True) Parameter containing:
tensor([[ 0.1470, -0.1385]], requires_grad=True)


The next Fibonacci sequence elements are : \[1597,2584,4181,6765,10946,17711\]

**Q4** Check that the RNN model is not overfitted.

In [12]:
fibonacci_test = np.array([377,610,987,1597,2584,4181,6765,10946,17711])
n_test = len(fibonacci_test) - n_steps #number of sequences
X_test = np.zeros((n_test,n_steps,d_inputs),dtype=np.float32)
y_test = np.zeros((n_test,d_inputs),dtype=np.float32)
for i in range(n_test):
    X_test[i,:,0] = fibonacci_test[i:i+n_steps]
    y_test[i] = fibonacci_test[i+n_steps]

batch_size=n_test
Xt_test = torch.from_numpy(X_test).reshape(batch_size, n_steps, d_inputs)  
yt_test = torch.from_numpy(y_test).reshape(batch_size, d_inputs)   

output_test=model(Xt_test)
print(output_test-yt_test)
print((output_test-yt_test)/yt_test)

tensor([[0.1209],
        [0.3137],
        [0.6292],
        [1.1377],
        [1.9614],
        [3.2939],
        [5.4512]], grad_fn=<SubBackward0>)
tensor([[0.0001],
        [0.0002],
        [0.0002],
        [0.0003],
        [0.0003],
        [0.0003],
        [0.0003]], grad_fn=<DivBackward0>)


<div style="border: 2px solid #4CAF50; padding: 10px; border-radius: 8px; background-color:#f0fff0">
The model generalizes to unseen Fibonacci values: the absolute/relative errors on the held-out continuation of the sequence remain very small (close to 0 compared to the target magnitudes). This indicates that the network learned the underlying recurrence $F_{t}=F_{t-1}+F_{t-2}$ rather than memorizing the training pairs.
</div>